### First Step : Clean raw data 

The dataset has been downloaded from the website ... . Some columns are completely useless (bookmakers infos) and some rows are full of missing values. The first step is to clean the dataset.

In [ ]:
import numpy as np
import pandas as pd
import numpy as np
import matplotlib as plt
from Data.data_cleaning import Datacleaner
from scipy.optimize import minimize
from scipy.special import expit
from BradleyTerry_classical.Gradient_descent import GradientDescent
from BradleyTerry_classical.BT_classical import BradleyTerry

ModuleNotFoundError: No module named 'Gradient_descent'

In [3]:
cleaner = Datacleaner("./Data/ligue1_2010_2025.csv")

cleaner = (cleaner.drop_useless_columns()
           .drop_rows(row_idxs=[1520,2607,2281])
           .fillNA(column='Div',value='F1')
           .fillNA(column='Time', value='00:00')
           .add_season())

cleaner.save_data("Data/Dataset_clean.csv")

/Users/paullemoinevandermoere/Desktop/ENSAE/ML 3A/Projet_ML/Learning-Latent-Football-Team-Strengths/Data/data_cleaning.py:25: DtypeWarning: Columns (139) have mixed types. Specify dtype option on import or set low_memory=False.
  self.df = pd.read_csv(filepath)


In [4]:
df = pd.read_csv("./Data/Dataset_clean.csv")
df.head(10)


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,AST,HC,AC,HF,AF,HY,AY,HR,AR,Season
0,F1,07/08/10,00:00,Auxerre,Lorient,2.0,2.0,D,1.0,1.0,...,2.0,3.0,3.0,14.0,9.0,3.0,3.0,0.0,0.0,10-11
1,F1,07/08/10,00:00,Lens,Nancy,1.0,2.0,A,0.0,1.0,...,6.0,2.0,4.0,17.0,18.0,3.0,1.0,0.0,1.0,10-11
2,F1,07/08/10,00:00,Lyon,Monaco,0.0,0.0,D,0.0,0.0,...,4.0,3.0,5.0,14.0,24.0,2.0,3.0,0.0,0.0,10-11
3,F1,07/08/10,00:00,Marseille,Caen,1.0,2.0,A,0.0,0.0,...,2.0,6.0,3.0,16.0,23.0,2.0,3.0,0.0,0.0,10-11
4,F1,07/08/10,00:00,Nice,Valenciennes,0.0,0.0,D,0.0,0.0,...,1.0,4.0,6.0,11.0,20.0,0.0,3.0,0.0,0.0,10-11
5,F1,07/08/10,00:00,Paris SG,St Etienne,3.0,1.0,H,2.0,1.0,...,5.0,6.0,7.0,11.0,12.0,2.0,1.0,0.0,0.0,10-11
6,F1,07/08/10,00:00,Rennes,Lille,1.0,1.0,D,1.0,0.0,...,5.0,6.0,3.0,23.0,21.0,3.0,3.0,0.0,0.0,10-11
7,F1,07/08/10,00:00,Sochaux,Arles,2.0,1.0,H,1.0,1.0,...,1.0,10.0,3.0,16.0,19.0,2.0,4.0,0.0,0.0,10-11
8,F1,07/08/10,00:00,Toulouse,Brest,2.0,0.0,H,1.0,0.0,...,1.0,2.0,3.0,21.0,18.0,3.0,2.0,0.0,0.0,10-11
9,F1,08/08/10,00:00,Montpellier,Bordeaux,1.0,0.0,H,0.0,0.0,...,1.0,5.0,1.0,22.0,13.0,5.0,2.0,0.0,0.0,10-11


### Second step : Classical Bradley-Terry

#### Calibrate the lambda hyperparameter using the whole dataset

In [4]:
# Process data to get the whole dataset
processer = DataProcesser("./Data/Dataset_clean.csv")

data = processer.get_data()

teams_all = data['Teams']
W_matrix_all = data['Victory Matrix']
D_matrix_all = data['Draw Matrix']


# Calibrate lambda
Lambda_Chooser = BT_calibrate(W= W_matrix_all, D= D_matrix_all, teams= teams_all)

lambda_grid = np.linspace(-3,2,30)

Lambda_Chooser.calibrate_lambda(lambda_grid=lambda_grid)

Season
10-11        380
11-12        380
12-13        380
13-14        380
14-15        380
15-16        380
17-18        380
2018-2019    380
2020-2021    380
2021-2022    380
2022-2023    380
16-17        379
2023-2024    306
2024-2025    306
2019-2020    279
2025-2026    144
Name: count, dtype: int64


#### Train the model with the calibrated lambda (-0.24)

In [ ]:
# Process data to get the wanted year (e.g., 2010-2011)
processer = DataProcesser("./Data/Dataset_clean.csv")

processer.filter_year(year='10-11')

train_df , test_df = processer.split_train_test(test_size=11, nb_teams=20)

train_data, test_data = processer.get_train_test_data()


teams = train_data['Teams']
W_matrix = train_data['Victory Matrix']
D_matrix = train_data['Draw Matrix']


# Train the model
bt_model = BradleyTerry(lambda_draw=-0.24, learning_rate=0.01, n_iterations=1000)
bt_model.fit(W=W_matrix, D=D_matrix, teams=teams)

strength = bt_model.predict_strength()

In [ ]:
result = {team: float(value) for team,value in zip(bt_model.teams,strength)}
result

In [ ]:
from Test.test_BT_classical import BT_Test

Test = BT_Test(model=bt_model, test_df=test_df)

Test.logLoss()

Test.baseline_logloss()

In [5]:
print("\n--- Categorical Statistics (Match Results) ---")
# Counts how many Home Wins (H), Draws (D), and Away Wins (A)
print(df['FTR'].value_counts())
# Visualizations 
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(15, 10))
plt.subplots_adjust(hspace=0.4, wspace=0.3) # Adjust spacing

# Plot A: match outcome distribution
counts = df['FTR'].value_counts()
axes[0,0].bar(counts.index, counts.values, color=["#012B7E", "#DAE4DD", "#E40F16"])
axes[0,0].set_title('Match Outcomes (FTR)')
axes[0,0].set_ylabel('Count')

# Plot B: goals distribution 
axes[0,1].hist(df['FTHG'].dropna(), bins=range(0,10), alpha=0.5, label='Home Goals', color='blue')
axes[0,1].hist(df['FTAG'].dropna(), bins=range(0,10), alpha=0.5, label='Away Goals', color='red')
axes[0,1].set_title('Distribution of Goals (Home vs Away)')
axes[0,1].set_xlabel('Goals Scored')
axes[0,1].legend()

# Plot C: shots comparison 
shots_data = [df['HS'].dropna(), df['AS'].dropna()]
axes[1,0].boxplot(shots_data, labels=['Home Shots', 'Away Shots'], patch_artist=True)
axes[1,0].set_title('Shot Volume: Home vs Away')
axes[1,0].set_ylabel('Number of Shots')

# Plot D: correlation matrix
numeric_df = df.select_dtypes(include=['number'])
corr = numeric_df.corr()
im = axes[1,1].imshow(corr, cmap='coolwarm', interpolation='nearest')
axes[1,1].set_title('Correlation Heatmap')
plt.colorbar(im, ax=axes[1,1])
tick_marks = range(len(corr.columns))
axes[1,1].set_xticks(tick_marks)
axes[1,1].set_yticks(tick_marks)
axes[1,1].set_xticklabels(corr.columns, rotation=45, ha='right')
axes[1,1].set_yticklabels(corr.columns)

plt.show()


--- Categorical Statistics (Match Results) ---
FTR
H    2476
A    1635
D    1483
Name: count, dtype: int64


AttributeError: module 'matplotlib' has no attribute 'subplots'

In [ ]:
df = df.dropna(subset=['HomeTeam', 'AwayTeam', 'FTR'])
all_teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).astype(str).unique()
all_teams.sort() # Now safe to sort
team_to_id = {name: i for i, name in enumerate(all_teams)} # Map team names to integers 
id_to_team = {i: name for name, i in team_to_id.items()}

# Convert 'FTR' (H/D/A) to Numeric Outcome (1.0, 0.5, 0.0)
def encode_outcome(ftr):
    if ftr == 'H': return 1.0   # Home Win
    if ftr == 'A': return 0.0   # Away Win
    return 0.5                  # Draw

df['Outcome_Num'] = df['FTR'].apply(encode_outcome)

# Create the matchups
matchups_data = df[['HomeTeam', 'AwayTeam', 'Outcome_Num']].copy()
matchups_data['HomeID'] = matchups_data['HomeTeam'].map(team_to_id)
matchups_data['AwayID'] = matchups_data['AwayTeam'].map(team_to_id)
matchups_array = matchups_data[['HomeID', 'AwayID', 'Outcome_Num']].values

In [ ]:
model = BradleyTerry(n_teams=len(all_teams), include_home_advantage=True)
model.fit(matchups_array)
rankings = model.get_rankings(id_to_team)

top_n = 18 #number of teams plotted
top_rankings = rankings[:top_n]
teams = [r[1] for r in top_rankings]
strengths = [r[2] for r in top_rankings]

plt.figure(figsize=(10, 8))
plt.barh(teams[::-1], strengths[::-1], color='skyblue')
plt.xlabel('Latent Strength (Theta)')
plt.title(f'Top {top_n} Teams by Bradley-Terry Strength')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.figtext(0.15, 0.02, f"Home Advantage: +{model.home_advantage:.3f}", 
            bbox={'facecolor':'yellow', 'alpha':0.2, 'pad':5})
plt.tight_layout()
plt.show()

In [ ]:
# Example: Predict last row of dataset
last_match = df.iloc[-4]
home, away = last_match['HomeTeam'], last_match['AwayTeam']
h_id, a_id = team_to_id[home], team_to_id[away]

prob_home_win = model.predict_prob(h_id, a_id, i_is_home=True)
print(f"Match: {home} vs {away}")
print(f"Predicted Probability of {home} winning: {prob_home_win:.2%}")

### Third step : Bradley-Terry Neural Network